# DF2WaveNet on UniMiB-SHAR

## Abstract

Evaluation of **DF2WaveNet** on the UniMiB-SHAR fall and activity dataset. Smartphone accelerometer windows (151×3) and `db2` wavelet features (151×12) are encoded through parallel TCN-LSTM branches and fused via attention for 17-class recognition. Results are reported over multiple random seeds.


## Introduction

HAR from body-worn inertial sensors requires models that capture both time-domain dynamics and multi-scale frequency structure. Convolutional and recurrent architectures often emphasize one representation at the expense of the other.

DF2WaveNet addresses this by:

1. Extracting multi-resolution wavelet features alongside the raw sensor stream.
2. Encoding each stream with shared TCN-LSTM backbones using dilated causal convolutions.
3. Fusing branch representations through attention before classification.

This notebook reproduces the full pipeline: data loading, wavelet preprocessing, model construction, training, and evaluation.


## Methodology

### Dual-stream representation

- **Stream 1 (raw):** Standardized accelerometer / inertial sequences.
- **Stream 2 (wavelet):** Per-sample `db2` discrete wavelet decomposition (level 3), concatenated across sub-bands along the feature axis.

### TCN-LSTM encoder (per stream)

Each stream passes through:

- Stacked TCN blocks with dilations `[1, 2, 4, 8, 16, 32, 64]`.
- WaveNet-style activation: `tanh(x) ⊙ σ(x)`.
- Channel normalization and residual connections.
- LSTM (32 units) → global average pooling → dense projection (128 units).

### Fusion and classification

Branch outputs are refined with self-attention, concatenated, batch-normalized, and classified through a two-layer dense head with softmax output.

### Training protocol

- Optimizer: Adam (`lr = 1×10⁻⁴`).
- Loss: sparse categorical crossentropy.
- Early stopping on validation accuracy (`patience = 40`, `restore_best_weights = True`).
- Multiple independent runs with distinct seeds; results reported as mean ± standard deviation.


## Experimental Setup

### Environment

- Python 3.10+
- TensorFlow 2.x, scikit-learn, PyWavelets, NumPy, pandas, matplotlib

### Reproducibility

- Random seeds are set per training run (NumPy, Python, TensorFlow).
- Stratified train/test splits use `random_state = 42` where applicable.
- Hyperparameters are centralized in the configuration cell below.

### Data paths

Set `DATA_ROOT` (and `WORK_DIR` for UniMiB-SHAR) to your local dataset location. Default paths target the Kaggle `har-sensor-datasets` layout.


### Dataset configuration

**Dataset:** UniMiB-SHAR — 11,771 accelerometer segments, 17 activity classes, 80/20 stratified split (`random_state=42`).

**Input shapes:** raw `(151, 3)`, wavelet `(151, 12)`.


In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pywt
import tensorflow as tf
from numpy import dstack
from pandas import read_csv
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from tensorflow.keras import Model, Input, backend as K, layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Activation,
    Add,
    BatchNormalization,
    Conv1D,
    Dense,
    Dropout,
    GlobalAveragePooling1D,
    Layer,
    LSTM,
    SpatialDropout1D,
)

print(f"TensorFlow {tf.__version__}")


In [ ]:
# ---------------------------------------------------------------------------
# Hyperparameters and runtime configuration
# ---------------------------------------------------------------------------

# --- Reproducibility ---
RANDOM_STATE = 42
N_RUNS = 2
SEEDS = [123, 456][:N_RUNS]

# --- Training ---
BATCH_SIZE = 32
EPOCHS = 400
LEARNING_RATE = 0.0001
EARLY_STOPPING_PATIENCE = 40
EARLY_STOPPING_MONITOR = "val_accuracy"

# --- Wavelet ---
WAVELET = "db2"
WAVELET_LEVEL = 3

# --- Model ---
LSTM_UNITS = 32
NB_FILTERS = 32
KERNEL_SIZE = 3
NUM_TCN_BLOCKS = 3
DROPOUT_RATE = 0.1
TCN_DILATIONS = [1, 2, 4, 8, 16, 32, 64]

# --- Performance toggles (optional GPU acceleration) ---
ULTRA_FAST = True
USE_MIXED_PRECISION = True
USE_XLA = True
CUDA_GROWTH = True
DISABLE_DETERMINISM_IN_FAST = True
USE_TF_DATA = True
PREFETCH_BUFFER = "AUTOTUNE"
CACHE_TRAIN_IN_MEMORY = True
NUM_PARALLEL_WAVELET = -1
VECTORIZE_SEQUENCES = True
VERBOSE_EPOCH_ONLY = 1
CLEAR_SESSION_BETWEEN_RUNS = True


def get_tf_optimizations() -> dict:
    """Enable mixed precision, XLA, and GPU memory growth when available."""
    opts: dict = {}
    if ULTRA_FAST and USE_MIXED_PRECISION:
        try:
            from tensorflow.keras import mixed_precision

            mixed_precision.set_global_policy(mixed_precision.Policy("mixed_float16"))
            opts["mixed_precision"] = True
        except Exception:
            opts["mixed_precision"] = False
    if ULTRA_FAST and USE_XLA:
        try:
            tf.config.optimizer.set_jit(True)
            opts["xla"] = True
        except Exception:
            opts["xla"] = False
    if ULTRA_FAST and CUDA_GROWTH:
        try:
            for gpu in tf.config.list_physical_devices("GPU"):
                tf.config.experimental.set_memory_growth(gpu, True)
            opts["memory_growth"] = True
        except Exception:
            opts["memory_growth"] = False
    return opts


def use_tf_data() -> bool:
    return ULTRA_FAST and USE_TF_DATA


def use_vectorized_sequences() -> bool:
    return ULTRA_FAST and VECTORIZE_SEQUENCES


def use_parallel_wavelet() -> bool:
    return ULTRA_FAST and NUM_PARALLEL_WAVELET != 0


def wavelet_joblib_n_jobs() -> int:
    return NUM_PARALLEL_WAVELET


def training_verbose() -> int:
    return VERBOSE_EPOCH_ONLY if ULTRA_FAST else 1


def clear_session_between_runs() -> bool:
    return ULTRA_FAST and CLEAR_SESSION_BETWEEN_RUNS


def use_determinism() -> bool:
    return not (ULTRA_FAST and DISABLE_DETERMINISM_IN_FAST)


def cache_train_dataset() -> bool:
    return ULTRA_FAST and CACHE_TRAIN_IN_MEMORY


def prefetch_buffer():
    try:
        return tf.data.AUTOTUNE if PREFETCH_BUFFER == "AUTOTUNE" else int(PREFETCH_BUFFER)
    except Exception:
        return 2


_tf_opts = get_tf_optimizations()
print(
    f"ULTRA_FAST={'ON' if ULTRA_FAST else 'OFF'} | "
    f"mixed_precision={_tf_opts.get('mixed_precision')} | "
    f"xla={_tf_opts.get('xla')} | "
    f"memory_growth={_tf_opts.get('memory_growth')}"
)

# ---------------------------------------------------------------------------
# Dataset: UniMiB-SHAR
# ---------------------------------------------------------------------------
DATA_ROOT = Path("/kaggle/input/unimib-shar/UniMiB-SHAR/data")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 151
N_AXES = 3
TEST_SIZE = 0.2
EXPECTED_SAMPLES = 11771

INPUT_SHAPE_RAW = (WINDOW_SIZE, N_AXES)
INPUT_SHAPE_WAVELET = (WINDOW_SIZE, 12)


## Implementation

### Data processing


In [ ]:
import scipy.io


def mat_to_csv_with_labels(acc_data_mat, acc_labels_mat, output_data_csv, output_labels_csv):
    """Convert UniMiB-SHAR .mat files to CSV format."""
    mat_data = scipy.io.loadmat(acc_data_mat)
    mat_labels = scipy.io.loadmat(acc_labels_mat)
    acc_data = mat_data["acc_data"]
    acc_labels = mat_labels["acc_labels"][:, 0]
    pd.DataFrame(acc_data).to_csv(output_data_csv, index=False, header=False)
    pd.DataFrame(acc_labels).to_csv(output_labels_csv, index=False, header=False)
    print(f"acc_data: {acc_data.shape} | labels: {len(acc_labels)} unique={len(pd.unique(acc_labels))}")


acc_data_csv = WORK_DIR / "acc_data.csv"
activity_labels_csv = WORK_DIR / "activity_labels.csv"
mat_to_csv_with_labels(
    DATA_ROOT / "acc_data.mat",
    DATA_ROOT / "acc_labels.mat",
    acc_data_csv,
    activity_labels_csv,
)

acc_data = pd.read_csv(acc_data_csv, header=None)
acc_labels = pd.read_csv(activity_labels_csv, header=None)


def reshape_and_split(acc_data, acc_labels, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """Reshape flat accelerometer readings to (samples, 151, 3) and stratified split."""
    data_values = acc_data.values
    labels = acc_labels.values.flatten()
    reshaped = data_values.reshape(-1, WINDOW_SIZE, N_AXES)
    if reshaped.shape[0] != EXPECTED_SAMPLES:
        raise ValueError(f"Expected {EXPECTED_SAMPLES} samples, got {reshaped.shape[0]}")
    return train_test_split(
        reshaped,
        labels[: reshaped.shape[0]],
        test_size=test_size,
        random_state=random_state,
        stratify=labels,
    )


X_train, X_test, y_train, y_test = reshape_and_split(acc_data, acc_labels)
trainy, testy = y_train, y_test
print(f"Train: {X_train.shape} | Test: {X_test.shape}")


In [ ]:
def _wavelet_one(args):
    """Worker for parallel wavelet decomposition."""
    i, x, wavelet, level = args
    try:
        coeffs = pywt.wavedec(x, wavelet, level=level, axis=1)
        return i, np.concatenate(coeffs, axis=1)
    except Exception:
        return i, None


def apply_wavelet_transform(X, wavelet=WAVELET, level=WAVELET_LEVEL):
    """Apply discrete wavelet decomposition to each sample.

    Args:
        X: Array of shape (n_samples, timesteps, n_features).
        wavelet: Wavelet family identifier.
        level: Decomposition depth.

    Returns:
        Wavelet feature array with concatenated sub-band coefficients.
    """
    use_par = use_parallel_wavelet()
    n_jobs = wavelet_joblib_n_jobs()

    if use_par and n_jobs != 0:
        from joblib import Parallel, delayed

        out = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(_wavelet_one)((i, X[i], wavelet, level)) for i in range(X.shape[0])
        )
        out = [o[1] for o in sorted(out, key=lambda x: x[0])]
        bad = [i for i, o in enumerate(out) if o is None]
        for i in bad:
            coeffs = pywt.wavedec(X[i], wavelet, level=level, axis=1)
            out[i] = np.concatenate(coeffs, axis=1)
        return np.array(out, dtype=np.float32)

    X_wavelet = []
    for i in range(X.shape[0]):
        coeffs = pywt.wavedec(X[i], wavelet, level=level, axis=1)
        X_wavelet.append(np.concatenate(coeffs, axis=1))
    return np.array(X_wavelet, dtype=np.float32)


X_train_wavelet = apply_wavelet_transform(X_train)
X_test_wavelet = apply_wavelet_transform(X_test)
print(f"Raw train/test: {X_train.shape} / {X_test.shape}")
print(f"Wavelet train/test: {X_train_wavelet.shape} / {X_test_wavelet.shape}")


In [ ]:
unique_labels = np.unique(np.concatenate((trainy, testy)))
num_classes = len(unique_labels)
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
trainy_mapped = np.array([label_mapping[label] for label in trainy])
testy_mapped = np.array([label_mapping[label] for label in testy])
print(f"Classes ({num_classes}): {unique_labels}")


In [ ]:
def scale_sequences(X_train, X_test, X_train_w, X_test_w):
    """Standardize raw and wavelet streams (per-feature, fit on train only)."""
    scaler = StandardScaler()
    n_train, timesteps, n_feat = X_train.shape

    X_tr = scaler.fit_transform(X_train.reshape(-1, n_feat))
    X_te = scaler.transform(X_test.reshape(-1, n_feat))
    X_train = X_tr.reshape(n_train, timesteps, n_feat)
    X_test = X_te.reshape(X_test.shape[0], timesteps, n_feat)

    _, _, n_feat_w = X_train_w.shape
    X_tr_w = scaler.fit_transform(X_train_w.reshape(-1, n_feat_w))
    X_te_w = scaler.transform(X_test_w.reshape(-1, n_feat_w))
    X_train_wavelet = X_tr_w.reshape(X_train_w.shape[0], timesteps, n_feat_w)
    X_test_wavelet = X_te_w.reshape(X_test_w.shape[0], timesteps, n_feat_w)
    return X_train, X_test, X_train_wavelet, X_test_wavelet


X_train, X_test, X_train_wavelet, X_test_wavelet = scale_sequences(
    X_train, X_test, X_train_wavelet, X_test_wavelet
)
print(f"Scaled raw: {X_train.shape} | Scaled wavelet: {X_train_wavelet.shape}")


### Model definition


In [ ]:
class ChannelNormalization(Layer):
    """Normalize each channel by its maximum absolute value."""

    def call(self, inputs):
        max_values = K.max(K.abs(inputs), axis=2, keepdims=True) + 1e-5
        return inputs / max_values


def wave_net_activation(x):
    """WaveNet gating: tanh(x) * sigmoid(x)."""
    tanh_out = Activation("tanh")(x)
    sigm_out = Activation("sigmoid")(x)
    return layers.multiply([tanh_out, sigm_out])


def dense_residual_block(x, s, i, nb_filters, kernel_size, dropout_rate=0):
    original_x = x
    conv = Conv1D(
        filters=nb_filters, kernel_size=kernel_size, dilation_rate=i, padding="causal"
    )(x)
    conv = BatchNormalization()(conv)
    x = wave_net_activation(conv)
    x = ChannelNormalization()(x)
    x = BatchNormalization()(x)
    x = SpatialDropout1D(dropout_rate)(x)

    if K.int_shape(original_x)[-1] != nb_filters:
        original_x = Conv1D(filters=nb_filters, kernel_size=1, padding="same")(original_x)
    return Add()([original_x, x])


def TCN_block(x, nb_filters, kernel_size, dilations, dropout_rate=0.1):
    for dilation_rate in dilations:
        x = dense_residual_block(
            x,
            s=0,
            i=dilation_rate,
            nb_filters=nb_filters,
            kernel_size=kernel_size,
            dropout_rate=dropout_rate,
        )
    return x


def TCN_LSTM_model(
    input_shape,
    lstm_units=LSTM_UNITS,
    nb_filters=NB_FILTERS,
    kernel_size=KERNEL_SIZE,
    num_tcn_blocks=NUM_TCN_BLOCKS,
    dropout_rate=DROPOUT_RATE,
):
    """Single-stream TCN-LSTM encoder."""
    inputs = Input(shape=input_shape)
    x = inputs
    for _ in range(num_tcn_blocks):
        x = TCN_block(
            x,
            nb_filters=nb_filters,
            kernel_size=kernel_size,
            dilations=TCN_DILATIONS,
            dropout_rate=dropout_rate,
        )
    x = LSTM(units=lstm_units, return_sequences=True)(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.2)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.2)(x)
    return Model(inputs, x)


class AttentionBlock(Layer):
    """Scaled dot-product attention with reduced-dimension residual path."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.dense_query = Dense(input_shape[-1])
        self.dense_key = Dense(input_shape[-1])
        self.dense_value = Dense(input_shape[-1])
        self.dense_residual = Dense(input_shape[-1] // 2)
        self.dense_residual_project = Dense(input_shape[-1])
        super().build(input_shape)

    def call(self, inputs):
        query = self.dense_query(inputs)
        key = self.dense_key(inputs)
        value = self.dense_value(inputs)
        scores = tf.nn.softmax(tf.matmul(query, key, transpose_b=True), axis=-1)
        attention_output = tf.matmul(scores, value)
        residual = self.dense_residual_project(self.dense_residual(inputs))
        return attention_output + residual


def create_fusion_model(
    input_shape_1,
    input_shape_2,
    num_classes,
    lstm_units=LSTM_UNITS,
    nb_filters=NB_FILTERS,
    kernel_size=KERNEL_SIZE,
    num_tcn_blocks=NUM_TCN_BLOCKS,
    dropout_rate=DROPOUT_RATE,
):
    """Dual-stream TCN-LSTM fusion model with attention."""
    branch_1 = TCN_LSTM_model(
        input_shape=input_shape_1,
        lstm_units=lstm_units,
        nb_filters=nb_filters,
        kernel_size=kernel_size,
        num_tcn_blocks=num_tcn_blocks,
        dropout_rate=dropout_rate,
    )
    branch_2 = TCN_LSTM_model(
        input_shape=input_shape_2,
        lstm_units=lstm_units,
        nb_filters=nb_filters,
        kernel_size=kernel_size,
        num_tcn_blocks=num_tcn_blocks,
        dropout_rate=dropout_rate,
    )

    attended_1 = AttentionBlock()(branch_1.output)
    attended_2 = AttentionBlock()(branch_2.output)
    combined = BatchNormalization()(layers.concatenate([attended_1, attended_2]))
    x = Dropout(0.2)(combined)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=[branch_1.input, branch_2.input], outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

input_shape_1 = INPUT_SHAPE_RAW
input_shape_2 = INPUT_SHAPE_WAVELET

model = create_fusion_model(input_shape_1, input_shape_2, num_classes)
model.summary()


## Results

Multi-seed training with early stopping. Metrics: accuracy, weighted recall, weighted F1, confusion matrix, per-class ROC-AUC.


In [ ]:
def set_seed(seed: int) -> None:
    """Set random seeds for a single experimental run."""
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    try:
        if use_determinism():
            tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def make_dataset(X1, X2, y, batch_size=BATCH_SIZE, shuffle=True, cache=False, prefetch_buf=2):
    """Build a tf.data pipeline for dual-input training."""
    ds = tf.data.Dataset.from_tensor_slices(((X1, X2), y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(4096, len(y)), reshuffle_each_iteration=True)
    if cache:
        ds = ds.cache()
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(prefetch_buf)
    return ds


run_accuracy, run_recall, run_f1 = [], [], []
history_last = None
predictions_last = None
model_last = None

use_ds = use_tf_data()
if use_ds:
    train_ds = make_dataset(
        X_train,
        X_train_wavelet,
        trainy_mapped,
        shuffle=True,
        cache=cache_train_dataset(),
        prefetch_buf=prefetch_buffer(),
    )
    val_ds = make_dataset(
        X_test,
        X_test_wavelet,
        testy_mapped,
        shuffle=False,
        cache=False,
        prefetch_buf=prefetch_buffer(),
    )

for run, seed in enumerate(SEEDS):
    if clear_session_between_runs() and run > 0:
        tf.keras.backend.clear_session()
    print(f"\nRun {run + 1}/{N_RUNS} (seed={seed})")
    set_seed(seed)

    model = create_fusion_model(input_shape_1, input_shape_2, num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    early_stopping = EarlyStopping(
        monitor=EARLY_STOPPING_MONITOR,
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    )
    verb = training_verbose()

    if use_ds:
        history = model.fit(
            train_ds,
            epochs=EPOCHS,
            validation_data=val_ds,
            callbacks=[early_stopping],
            verbose=verb,
        )
    else:
        history = model.fit(
            [X_train, X_train_wavelet],
            trainy_mapped,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=([X_test, X_test_wavelet], testy_mapped),
            callbacks=[early_stopping],
            verbose=verb,
        )

    history_last = history
    model_last = model

    predictions = model.predict([X_test, X_test_wavelet], verbose=0)
    predictions_last = predictions
    predicted_classes = np.argmax(predictions, axis=1)

    acc = accuracy_score(testy_mapped, predicted_classes)
    rec = recall_score(testy_mapped, predicted_classes, average="weighted", zero_division=1)
    f1 = f1_score(testy_mapped, predicted_classes, average="weighted", zero_division=1)
    run_accuracy.append(acc)
    run_recall.append(rec)
    run_f1.append(f1)
    print(f"  Accuracy={acc:.4f}  Recall={rec:.4f}  F1={f1:.4f}")

print("\n" + "=" * 60)
print(f"RESULTS OVER {N_RUNS} RUNS (mean ± std)")
print("=" * 60)
print(f"Accuracy: {np.mean(run_accuracy):.4f} ± {np.std(run_accuracy):.4f}")
print(f"Recall:   {np.mean(run_recall):.4f} ± {np.std(run_recall):.4f}")
print(f"F1 Score: {np.mean(run_f1):.4f} ± {np.std(run_f1):.4f}")

for r in range(N_RUNS):
    print(
        f"  Run {r + 1}: Acc={run_accuracy[r]:.4f}, "
        f"Recall={run_recall[r]:.4f}, F1={run_f1[r]:.4f}"
    )

# --- Visualization (last run) ---
history = history_last
predictions = predictions_last
predicted_classes = np.argmax(predictions, axis=1)

plt.figure()
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")
plt.title(f"Training curves (seed={SEEDS[-1]})")
plt.show()

print("\nConfusion matrix:")
print(confusion_matrix(testy_mapped, predicted_classes))
print("\nClassification report:")
print(classification_report(testy_mapped, predicted_classes))

n_cls = predictions.shape[1]
Y_test_bin = label_binarize(testy_mapped, classes=np.arange(n_cls))
plt.figure()
for i in range(n_cls):
    if np.sum(Y_test_bin[:, i]) > 0:
        fpr, tpr, _ = roc_curve(Y_test_bin[:, i], predictions[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f"Class {i} (AUC={roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves (last run)")
plt.legend(loc="lower right")
plt.show()


## Analysis

Compare training and validation curves for overfitting. Inspect the confusion matrix for class pairs with elevated confusion. Per-class ROC curves highlight sensitivity–specificity trade-offs across activity categories.

Results should be interpreted in the context of dataset-specific class imbalance and windowing strategy.


## Conclusion

This notebook documents the DF2WaveNet dual-stream fusion pipeline end to end. The wavelet branch supplies multi-scale features complementary to raw inertial inputs; TCN-LSTM encoders and attention-based fusion integrate both representations for HAR classification.


